<a href="https://colab.research.google.com/github/roeiyanku/UAV_Sound_Classification_Project/blob/main/notebooks/03_train_classifiers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Training

This notebook uses the audio features extracted earlier and trains different
anomaly-detection models. **Changes vs v1:**

1. Saves a full per-run report (HTML + figures + CSV + JSON) with a
   timestamped folder name including the **hour, minute, second** — not just date.
2. Confusion matrices are saved to disk (and embedded in the HTML report)
   instead of just being shown inline.
3. Adds **ROC** and **Precision–Recall** curves per model, plus overlay plots.
4. Fixes a labeling bug: `target_names=["Normal","Anomaly"]` in v1 was mapping
   the names in the **wrong** order (your labels are `{abnormal: 0, normal: 1}`,
   so sklearn's report put abnormal under "Normal"). The v2 report indexes
   metrics by integer label and is explicit about which class is the anomaly.
5. Reports anomaly-class metrics consistently for **all** models (classical
   ML, CNN, teacher-student) — v1 mixed normal-class and anomaly-class F1.

Drop `reporting.py` next to this notebook (or upload it to `MODELS_DIR`).


## 1) Paths & Constants

In [ ]:
import os
from google.colab import drive

MOUNT = "/content/drive"

if os.path.ismount(MOUNT):
    print("Drive already mounted, skipping.")
else:
    # Nuke stale leftovers from a previous session
    if os.path.exists(MOUNT):
        !rm -rf {MOUNT}
    drive.mount(MOUNT)

Mounted at /content/drive


In [ ]:
import os
import joblib

# ── All paths and settings are defined here ──────────────────────────────────
MODELS_DIR    = "/content/drive/MyDrive/Final Project RMOT/artifacts/models"
RESULTS_DIR   = "/content/drive/MyDrive/Final Project RMOT/artifacts/results"
REPORTING_DIR = MODELS_DIR

USE_AUGMENTED = False
DATASET = "slider"
DB_LEVEL = "0_dB"
ID = "id_00"

if USE_AUGMENTED:
    PROCESSED_DIR = f"/content/drive/MyDrive/Final Project RMOT/processed/augmented/{DATASET}/{DB_LEVEL}/{ID}"
    SPLITS_DIR    = PROCESSED_DIR
else:
    PROCESSED_DIR = f"/content/drive/MyDrive/Final Project RMOT/processed/features/{DATASET}/{DB_LEVEL}/{ID}"
    SPLITS_DIR    = f"/content/drive/MyDrive/Final Project RMOT/processed/splits/{DATASET}/{DB_LEVEL}/{ID}"

os.makedirs(RESULTS_DIR, exist_ok=True)

# ── DYNAMIC CLASS ENCODING ALIGNMENT (Fixes Label Inversion) ─────────────────
splits_path = os.path.join(SPLITS_DIR, "processed_audio_splits.joblib")

try:
    data_splits = joblib.load(splits_path)
    label_to_int = data_splits.get('label_to_int', {})
    unique_labels = data_splits.get('unique_labels', [])
    print(f" Loaded metadata from splits file. Detected mapping: {label_to_int}")
except Exception as e:
    print(f" Could not load splits metadata automatically ({e}). Using safety fallback.")
    label_to_int = {}
    unique_labels = []

# ── Assign ANOMALY_LABEL and LABEL_NAMES based on actual content ───────────
if 'yes aircraft' in label_to_int:
    # Alphabetical order: "No aircraft"=0 (normal), "yes aircraft"=1 (anomaly)
    ANOMALY_LABEL = label_to_int['yes aircraft']  # Evaluates dynamically to 1
    LABEL_NAMES = ("normal", "anomaly")           # Tuple ordered [Index 0 = normal, Index 1 = anomaly]

elif 'drone' in label_to_int:
    # Alphabetical order: "drone"=0 (anomaly), "unknown"=1 (normal)
    ANOMALY_LABEL = label_to_int['drone']         # Evaluates dynamically to 0
    LABEL_NAMES = ("anomaly", "normal")           # Tuple ordered [Index 0 = anomaly, Index 1 = normal]

elif 'yes_drone' in label_to_int:
    # Alphabetical order: "unknown"=0 (normal), "yes_drone"=1 (anomaly)
    ANOMALY_LABEL = label_to_int['yes_drone']     # Evaluates dynamically to 1
    LABEL_NAMES = ("normal", "anomaly")

elif 'abnormal' in label_to_int:
    # Alphabetical order: "abnormal"=0 (anomaly), "normal"=1 (normal)
    ANOMALY_LABEL = label_to_int['abnormal']      # Evaluates dynamically to 0
    LABEL_NAMES = ("anomaly", "normal")

else:
    print("Standard dataset template not detected. Defaulting to standard binary structure.")
    ANOMALY_LABEL = 1
    LABEL_NAMES = ("normal", "anomaly")

print(f"Pipeline dynamically configured:")
print(f"   -> Target Anomaly Integer Class: {ANOMALY_LABEL}")
print(f"   -> Tuple mapping structure: {LABEL_NAMES} (Index {ANOMALY_LABEL} is treated as Anomaly)\n")


# ── Model hyperparameters ─────────────────────────────────────────────────────
CNN_EPOCHS        = 20
CNN_BATCH_SIZE    = 32
CNN_PATIENCE      = 5

TS_EPOCHS         = 20
TS_BATCH_SIZE     = 32
TS_PATIENCE       = 5

OCSVM_NU          = 0.1
ISO_N_ESTIMATORS  = 300
ISO_CONTAMINATION = 0.03
LOGREG_MAX_ITER   = 1000

RANDOM_STATE      = 42

 Loaded metadata from splits file. Detected mapping: {'abnormal': 0, 'normal': 1}
Pipeline dynamically configured:
   -> Target Anomaly Integer Class: 0
   -> Tuple mapping structure: ('anomaly', 'normal') (Index 0 is treated as Anomaly)



## 2) Load extracted features and reporting module

In [ ]:
!pip install weasyprint --quiet

import sys, os, time, copy, importlib
import joblib
import numpy as np
import pandas as pd

# remove old cached import files
!rm -rf "/content/drive/MyDrive/Final Project RMOT/artifacts/models/__pycache__"

# force Python to search this folder
sys.path.insert(0, MODELS_DIR)
importlib.invalidate_caches()

sys.path.append(MODELS_DIR)

from sklearn.metrics import accuracy_score
from reporting import TrainingReport

wav2vec_features  = joblib.load(os.path.join(PROCESSED_DIR, "extracted_wav2vec_features.joblib"))
ast_features      = joblib.load(os.path.join(PROCESSED_DIR, "extracted_ast_features.joblib"))
#logmel_features   = joblib.load(os.path.join(PROCESSED_DIR, "extracted_logmel_features.joblib"))
yamnet_features   = joblib.load(os.path.join(PROCESSED_DIR, "extracted_yamnet_features.joblib"))
yamnet_triplet_features = joblib.load(os.path.join(PROCESSED_DIR, "extracted_yamnet_triplet_features.joblib"))
ast_triplet_features = joblib.load(os.path.join(PROCESSED_DIR, "extracted_triplet_ast_features.joblib"))
labels            = joblib.load(os.path.join(PROCESSED_DIR, "labels.joblib"))

print("All feature files loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 29.7 MB/s eta 0:00:00
All feature files loaded.


In [ ]:

from models import (
    get_ocsvm, get_isolation_forest, train_model, predict_model,
    get_anomaly_scores, build_cnn_model,
    build_teacher_student_models, train_teacher_student, predict_teacher_student,
    get_logistic_regression
)

y_train = np.asarray(labels["train"])
y_val   = np.asarray(labels["val"])
y_test  = np.asarray(labels["test"])


## 3) Initialise the report

Creates a timestamped run folder under `RESULTS_DIR`.

In [ ]:
hyperparams = {
    "USE_AUGMENTED":     USE_AUGMENTED,
    "DATASET":           DATASET,
    "RANDOM_STATE":      RANDOM_STATE,
    "OCSVM_NU":          OCSVM_NU,
    "ISO_N_ESTIMATORS":  ISO_N_ESTIMATORS,
    "ISO_CONTAMINATION": ISO_CONTAMINATION,
    "LOGREG_MAX_ITER":   LOGREG_MAX_ITER,
    "CNN_EPOCHS":        CNN_EPOCHS,
    "CNN_BATCH_SIZE":    CNN_BATCH_SIZE,
    "CNN_PATIENCE":      CNN_PATIENCE,
    "TS_EPOCHS":         TS_EPOCHS,
    "TS_BATCH_SIZE":     TS_BATCH_SIZE,
    "TS_PATIENCE":       TS_PATIENCE,
}

report = TrainingReport(
    root_dir      = RESULTS_DIR,
    dataset       = DATASET,
    augmented     = USE_AUGMENTED,
    anomaly_label = ANOMALY_LABEL,
    label_names   = LABEL_NAMES,
    hyperparams   = hyperparams,
    extra_metadata={"models_dir": MODELS_DIR, "processed_dir": PROCESSED_DIR},
)
report.log_data_summary(y_train, y_val, y_test)
print("Run folder:", report.run_dir)


Run folder: /content/drive/MyDrive/Final Project RMOT/artifacts/results/run_2026-06-01_07-01-05_slider


## 4) Configure which features × models to train

In [ ]:
features = {
    "wav2vec": wav2vec_features,
    "ast":     ast_features,
    #"logmel":  logmel_features,
    "yamnet":  yamnet_features,
    "yamnet_triplet": yamnet_triplet_features,
    "ast_triplet": ast_triplet_features
}

models = {
    "ocsvm":      get_ocsvm(nu=OCSVM_NU),
    "iso_forest": get_isolation_forest(n_estimators=ISO_N_ESTIMATORS, contamination=ISO_CONTAMINATION),
    "logreg":     get_logistic_regression(max_iter=LOGREG_MAX_ITER),
}

SELECTED_FEATURES = ["ast","ast_triplet","wav2vec", "yamnet", "yamnet_triplet"]
SELECTED_MODELS   = ["iso_forest", "ocsvm"]

RUN_CNN             = False
RUN_TEACHER_STUDENT = False


## 5) Train classical models (OCSVM / IsoForest / LogReg)

Score-direction note for `add_result(scores=..., scores_higher_is_anomaly=...)`:

| Model       | What `scores` is                              | Higher = anomaly? |
|-------------|-----------------------------------------------|-------------------|
| ocsvm       | `model.decision_function(X)` — inlier margin  | **No** (higher = more normal) |
| iso_forest  | `model.score_samples(X)` — anomaly score (high = normal in sklearn) | **No** |
| logreg      | `model.predict_proba(X)[:, 1]` = P(class=1)=P(normal) | **No** |




In [ ]:
UNSUPERVISED_MODELS = {"ocsvm", "iso_forest"}
SUPERVISED_MODELS   = {"logreg"}

# Calculate normal mask based on aligned size of y_train
normal_mask = (y_train == (1 - ANOMALY_LABEL))   # 1 if sample is normal

for feature_name in SELECTED_FEATURES:

    # ── 1. DYNAMICALLY DETECT AND LOAD TRAINING KEY ───────────────────────────
    plural_train_key = f"{feature_name}_train_features"
    singular_train_key = f"{feature_name}_train_feature"

    if plural_train_key in features[feature_name]:
        X_train_data = features[feature_name][plural_train_key]
    elif singular_train_key in features[feature_name]:
        X_train_data = features[feature_name][singular_train_key]
    else:
        raise KeyError(
            f"Could not find a valid training feature key inside features['{feature_name}']. "
            f"Available keys: {list(features[feature_name].keys())}"
        )

    # ── 2. DYNAMICALLY DETECT AND LOAD TESTING KEY ────────────────────────────
    plural_test_key = f"{feature_name}_test_features"
    singular_test_key = f"{feature_name}_test_feature"

    if plural_test_key in features[feature_name]:
        X_test_data = features[feature_name][plural_test_key]
    elif singular_test_key in features[feature_name]:
        X_test_data = features[feature_name][singular_test_key]
    else:
        raise KeyError(
            f"Could not find a valid testing feature key inside features['{feature_name}']. "
            f"Available keys: {list(features[feature_name].keys())}"
        )

    # ── 3. DYNAMIC ALIGNMENT SQUEEZE (Fixes IndexError) ──────────────────────
    if X_train_data.shape[0] != len(y_train):
        print(f"Row mismatch on {feature_name}: Feature rows ({X_train_data.shape[0]}) != y_train ({len(y_train)}). Slicing features to align.")
        X_train_data = X_train_data[:len(y_train)]

    if X_test_data.shape[0] != len(y_test):
        X_test_data = X_test_data[:len(y_test)]

    # This boolean index mask will now securely match your indexed array boundaries
    X_train_normal = X_train_data[normal_mask]

    # ── 4. RUN MODEL TRAINING AND EVALUATION SUITE ────────────────────────────
    for model_name in SELECTED_MODELS:
        print(f"\nTraining {model_name} on {feature_name}...")
        model = copy.deepcopy(models[model_name])

        if model_name in UNSUPERVISED_MODELS:
            t_train0 = time.time()
            model = train_model(model, X_train_normal)
            train_time_s = time.time() - t_train0

            t0 = time.time()
            raw_preds = predict_model(model, X_test_data)  # +1 inlier, -1 outlier
            latency_s = time.time() - t0

            # Map sklearn's {+1=inlier, -1=outlier} -> our {1=normal, 0=anomaly}
            preds = np.where(raw_preds == 1, 1 - ANOMALY_LABEL, ANOMALY_LABEL)

            if model_name == "ocsvm":
                scores = model.decision_function(X_test_data)
            elif model_name == "iso_forest":
                scores = model.score_samples(X_test_data)
            else:
                scores = None
            scores_higher_is_anomaly = False  # both metrics: higher = more normal

        elif model_name in SUPERVISED_MODELS:
            t_train0 = time.time()
            model.fit(X_train_data, y_train)
            train_time_s = time.time() - t_train0

            t0 = time.time()
            preds = model.predict(X_test_data)
            latency_s = time.time() - t0

            try:
                proba = model.predict_proba(X_test_data)
                # P(class=1) where class 1 is the normal class
                scores = proba[:, 1]
                scores_higher_is_anomaly = False
            except Exception:
                scores = None
                scores_higher_is_anomaly = True

        else:
            print(f"Skipping unsupported model: {model_name}")
            continue

        # Save model checkpoint safely
        ckpt_path = os.path.join(MODELS_DIR, f"{feature_name}_{model_name}.joblib")
        joblib.dump(model, ckpt_path)

        result = report.add_result(
            feature_name = feature_name,
            model_name   = model_name,
            y_true       = y_test,
            y_pred       = preds,
            scores       = scores,
            scores_higher_is_anomaly = scores_higher_is_anomaly,
            train_time_s = train_time_s,
            latency_total_s = latency_s,
        )
        print(f"  Acc {result.accuracy:.3f} | "
              f"F1(anom) {result.f1_anomaly:.3f} | "
              f"AUC {result.auc if result.auc is None else f'{result.auc:.3f}'} | "
              f"{result.latency_ms_sample:.3f} ms/sample")


Training iso_forest on ast...
  Acc 0.963 | F1(anom) 0.927 | AUC 0.991 | 0.151 ms/sample

Training ocsvm on ast...
  Acc 0.907 | F1(anom) 0.844 | AUC 0.996 | 0.063 ms/sample

Training iso_forest on ast_triplet...
  Acc 0.972 | F1(anom) 0.947 | AUC 1.000 | 0.193 ms/sample

Training ocsvm on ast_triplet...
  Acc 0.939 | F1(anom) 0.893 | AUC 1.000 | 0.022 ms/sample

Training iso_forest on wav2vec...
  Acc 0.790 | F1(anom) 0.366 | AUC 0.902 | 0.229 ms/sample

Training ocsvm on wav2vec...
  Acc 0.827 | F1(anom) 0.648 | AUC 0.830 | 0.057 ms/sample

Training iso_forest on yamnet...
  Acc 0.836 | F1(anom) 0.598 | AUC 0.941 | 0.132 ms/sample

Training ocsvm on yamnet...
  Acc 0.897 | F1(anom) 0.825 | AUC 0.974 | 0.060 ms/sample

Training iso_forest on yamnet_triplet...
  Acc 0.935 | F1(anom) 0.885 | AUC 0.998 | 0.141 ms/sample

Training ocsvm on yamnet_triplet...
  Acc 0.893 | F1(anom) 0.824 | AUC 1.000 | 0.012 ms/sample


## 6) CNN on log-mel

In [ ]:
if RUN_CNN and "logmel" in features:
    from tensorflow import keras

    X_train_cnn = np.asarray(features["logmel"]["logmel_train_features"], dtype=np.float32)
    X_val_cnn   = np.asarray(features["logmel"]["logmel_val_features"],   dtype=np.float32)
    X_test_cnn  = np.asarray(features["logmel"]["logmel_test_features"],  dtype=np.float32)

    if X_train_cnn.ndim == 3:
        X_train_cnn = X_train_cnn[..., np.newaxis]
        X_val_cnn   = X_val_cnn[..., np.newaxis]
        X_test_cnn  = X_test_cnn[..., np.newaxis]

    cnn_model = build_cnn_model(input_shape=X_train_cnn.shape[1:])

    es = keras.callbacks.EarlyStopping(monitor="val_loss",
                                       patience=CNN_PATIENCE, restore_best_weights=True)
    mc = keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, "cnn_logmel_best.keras"),
        monitor="val_loss", save_best_only=True,
    )

    t0 = time.time()
    cnn_model.fit(
        X_train_cnn, y_train,
        validation_data=(X_val_cnn, y_val),
        epochs=CNN_EPOCHS, batch_size=CNN_BATCH_SIZE,
        callbacks=[es, mc], verbose=1,
    )
    train_time_s = time.time() - t0

    t0 = time.time()
    probs = cnn_model.predict(X_test_cnn).ravel()
    latency_s = time.time() - t0
    # The CNN was trained with y_train where class 1 = normal,
    # so its sigmoid output = P(normal). Higher = more normal.
    preds = (probs >= 0.5).astype(int)

    result = report.add_result(
        feature_name = "logmel",
        model_name   = "cnn",
        y_true       = y_test,
        y_pred       = preds,
        scores       = probs,
        scores_higher_is_anomaly = False,   # P(normal); higher means less anomalous
        train_time_s = train_time_s,
        latency_total_s = latency_s,
        model_hyperparams = {"epochs": CNN_EPOCHS, "batch_size": CNN_BATCH_SIZE,
                             "patience": CNN_PATIENCE},
        notes = "Sigmoid output of class=1 (normal).",
    )
    print(f"CNN -> Acc {result.accuracy:.3f} | F1(anom) {result.f1_anomaly:.3f} | "
          f"AUC {result.auc:.3f}")


## 7) Teacher–student on log-mel

In [ ]:
if RUN_TEACHER_STUDENT and "logmel" in features:
    X_train_lm = features["logmel"]["logmel_train_features"]
    X_test_lm  = features["logmel"]["logmel_test_features"]

    X_train_flat = X_train_lm.reshape(len(X_train_lm), -1)
    X_test_flat  = X_test_lm.reshape(len(X_test_lm),   -1)
    X_train_normal = X_train_flat[normal_mask]

    teacher, student = build_teacher_student_models(input_dim=X_train_flat.shape[1])

    t0 = time.time()
    student = train_teacher_student(
        teacher, student, X_train_flat, X_train_normal,
        epochs=TS_EPOCHS, batch_size=TS_BATCH_SIZE, patience=TS_PATIENCE,
    )
    train_time_s = time.time() - t0
    student.save(os.path.join(MODELS_DIR, "teacher_student_logmel.keras"))

    t0 = time.time()
    preds, scores, threshold = predict_teacher_student(
        teacher, student, X_train_normal, X_test_flat
    )
    latency_s = time.time() - t0

    # `predict_teacher_student` produces an anomaly score (teacher-student
    # distance), so higher = more anomalous.
    result = report.add_result(
        feature_name = "logmel",
        model_name   = "teacher_student",
        y_true       = y_test,
        y_pred       = preds,
        scores       = scores,
        scores_higher_is_anomaly = True,
        train_time_s = train_time_s,
        latency_total_s = latency_s,
        model_hyperparams = {"epochs": TS_EPOCHS, "batch_size": TS_BATCH_SIZE,
                             "patience": TS_PATIENCE},
        notes = f"Threshold (95th pct of train scores): {threshold:.4f}",
    )
    print(f"TS -> Acc {result.accuracy:.3f} | F1(anom) {result.f1_anomaly:.3f} | "
          f"AUC {result.auc:.3f}")


## 8) Finalise the report

Writes `results.csv`, `results.json`, `metadata.json`, `report.html`, and per-model figures into the run folder.

In [ ]:
pdf_path = report.finalize()
print("PDF report saved to:", pdf_path)
print("\nRun folder contents:")
for root, dirs, files in os.walk(report.run_dir):
    rel = os.path.relpath(root, report.run_dir)
    for f in sorted(files):
        print(f"  {os.path.join(rel, f)}")


DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.002s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.007s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'glyf' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'glyf' tabl

PDF report saved to: /content/drive/MyDrive/Final Project RMOT/artifacts/results/run_2026-06-01_07-01-05_slider/report.pdf

Run folder contents:
  ./metadata.json
  ./report.pdf
  ./results.csv
  ./results.json
  figures/cm_ast_iso_forest.png
  figures/cm_ast_ocsvm.png
  figures/cm_ast_triplet_iso_forest.png
  figures/cm_ast_triplet_ocsvm.png
  figures/cm_wav2vec_iso_forest.png
  figures/cm_wav2vec_ocsvm.png
  figures/cm_yamnet_iso_forest.png
  figures/cm_yamnet_ocsvm.png
  figures/cm_yamnet_triplet_iso_forest.png
  figures/cm_yamnet_triplet_ocsvm.png
  figures/pr_ast_iso_forest.png
  figures/pr_ast_ocsvm.png
  figures/pr_ast_triplet_iso_forest.png
  figures/pr_ast_triplet_ocsvm.png
  figures/pr_overlay.png
  figures/pr_wav2vec_iso_forest.png
  figures/pr_wav2vec_ocsvm.png
  figures/pr_yamnet_iso_forest.png
  figures/pr_yamnet_ocsvm.png
  figures/pr_yamnet_triplet_iso_forest.png
  figures/pr_yamnet_triplet_ocsvm.png
  figures/roc_ast_iso_forest.png
  figures/roc_ast_ocsvm.png
  figures

In [ ]:
# Quick on-screen view of the headline table
df = report.to_dataframe()
display_cols = ["feature_name", "model_name", "accuracy",
                "precision_anomaly", "recall_anomaly", "f1_anomaly",
                "auc", "average_precision",
                "train_time_s", "latency_ms_sample"]
df[display_cols].sort_values(by=["auc", "f1_anomaly"], ascending=False)


,feature_name,model_name,accuracy,precision_anomaly,recall_anomaly,f1_anomaly,auc,average_precision,train_time_s,latency_ms_sample
2,ast_triplet,iso_forest,0.971963,0.900000,1.000000,0.947368,1.000000,1.000000,0.858275,0.193452
3,ast_triplet,ocsvm,0.939252,0.805970,1.000000,0.892562,1.000000,1.000000,0.013745,0.021586
9,yamnet_triplet,ocsvm,0.892523,0.701299,1.000000,0.824427,1.000000,1.000000,0.010945,0.012418
8,yamnet_triplet,iso_forest,0.934579,0.794118,1.000000,0.885246,0.998264,0.994165,0.579483,0.141084
1,ast,ocsvm,0.906542,0.729730,1.000000,0.843750,0.996296,0.986599,0.042452,0.062754
0,ast,iso_forest,0.962617,0.910714,0.944444,0.927273,0.991088,0.966679,0.704451,0.151392
7,yamnet,ocsvm,0.897196,0.722222,0.962963,0.825397,0.973727,0.939688,0.058483,0.060401
6,yamnet,iso_forest,0.836449,0.787879,0.481481,0.597701,0.941088,0.808498,0.582114,0.132370
4,wav2vec,iso_forest,0.789720,0.764706,0.240741,0.366197,0.901736,0.718698,1.014515,0.228673
5,wav2vec,ocsvm,0.827103,0.666667,0.629630,0.647619,0.830093,0.686836,0.041232,0.056903


In [ ]:
from google.colab import drive
drive.flush_and_unmount()